# Building an LP Model from Scratch

**Goal:** Implement the core indexing and constraint-building functions for a 252-variable, 48-constraint linear program.

**Time budget:** 60-75 minutes

## Problem recap

We're optimizing milk sourcing across 3 plants over 12 months.

### Decision variables (252 total)

- **x[p, s, t]** — volume bought from spot supplier s at plant p in month t  
  (3 plants × 3 suppliers × 12 months = 108 variables)

- **u[r, t]** — volume transferred on route r in month t  
  (6 routes × 12 months = 72 variables)

- **e[p, k, t]** — excess milk sold from plant p to customer k in month t  
  (3 plants × 2 customers × 12 months = 72 variables)

### Constraints

- **36 balance constraints** — net inflow = production need at each (plant, month)
- **12 S3 capacity constraints** — total S3 purchases ≤ 150,000 L per month

### The indexing challenge

`scipy.linprog` expects a **flat 1D vector** of 252 variables. We need helper functions to map logical coordinates (plant, supplier, month) → integer index.

In [ ]:
# Setup
import numpy as np

from milk_balancing.constants import NK, NP, NS, ROUTES, T

print("Problem dimensions:")
print(f"  Plants: {NP}")
print(f"  Spot suppliers: {NS}")
print(f"  Customers: {NK}")
print(f"  Months: {T}")
print(f"  Transfer routes: {len(ROUTES)}")
print(f"  Routes: {ROUTES}")
print(f"\nTotal variables: {NP * NS * T + len(ROUTES) * T + NP * NK * T}")

---

## Part 1: Variable Indexing (20 min)

### Context

We organize the 252 variables in three consecutive blocks:

```
[ x[0,0,0] ... x[2,2,11] | u[route0,0] ... u[route5,11] | e[0,0,0] ... e[2,1,11] ]
  indices 0-107          | indices 108-179              | indices 180-251
```

### Your task

Implement the three indexing functions below. Each maps logical coordinates to a flat index.

**Hint:** For `idx_x`, think about how to flatten a 3D array with shape (NP, NS, T) into 1D.

In [ ]:
# YOUR CODE HERE


def idx_x(p: int, s: int, t: int) -> int:
    """Flat index of spot-purchase variable x[p, s, t].  Range: 0 … 107."""
    raise NotImplementedError("Implement idx_x() function.")


def idx_u(r: tuple, t: int) -> int:
    """Flat index of transfer variable u[r, t].  Range: 108 … 179."""
    raise NotImplementedError("Implement idx_u() function.")


def idx_e(p: int, k: int, t: int) -> int:
    """Flat index of excess-sales variable e[p, k, t].  Range: 180 … 251."""
    raise NotImplementedError("Implement idx_e() function.")

### Test your implementation

In [ ]:
# Basic boundary tests
assert idx_x(0, 0, 0) == 0, "First x variable should be at index 0"
assert idx_x(2, 2, 11) == 107, "Last x variable should be at index 107"

assert idx_u(ROUTES[0], 0) == 108, "First u variable should be at index 108"
assert idx_u(ROUTES[-1], 11) == 179, "Last u variable should be at index 179"

assert idx_e(0, 0, 0) == 180, "First e variable should be at index 180"
assert idx_e(2, 1, 11) == 251, "Last e variable should be at index 251"

print("✅ Boundary tests passed!")

# Uniqueness test - make sure no collisions
all_indices = set()

for p in range(NP):
    for s in range(NS):
        for t in range(T):
            idx = idx_x(p, s, t)
            assert 0 <= idx <= 107, f"idx_x({p},{s},{t}) = {idx} out of range"
            assert idx not in all_indices, f"Duplicate index {idx}"
            all_indices.add(idx)

for r in ROUTES:
    for t in range(T):
        idx = idx_u(r, t)
        assert 108 <= idx <= 179, f"idx_u({r},{t}) = {idx} out of range"
        assert idx not in all_indices, f"Duplicate index {idx}"
        all_indices.add(idx)

for p in range(NP):
    for k in range(NK):
        for t in range(T):
            idx = idx_e(p, k, t)
            assert 180 <= idx <= 251, f"idx_e({p},{k},{t}) = {idx} out of range"
            assert idx not in all_indices, f"Duplicate index {idx}"
            all_indices.add(idx)

assert len(all_indices) == 252, f"Expected 252 unique indices, got {len(all_indices)}"
print("✅ All 252 indices are unique and within correct ranges!")

<details>
<summary><b>💡 SOLUTION</b> (click to expand — try on your own first!)</summary>

```python
def idx_x(p: int, s: int, t: int) -> int:
    """Flat index of spot-purchase variable x[p, s, t].  Range: 0 … 107."""
    return p * NS * T + s * T + t


def idx_u(r: tuple, t: int) -> int:
    """Flat index of transfer variable u[r, t].  Range: 108 … 179."""
    return NP * NS * T + ROUTES.index(r) * T + t


def idx_e(p: int, k: int, t: int) -> int:
    """Flat index of excess-sales variable e[p, k, t].  Range: 180 … 251."""
    return NP * NS * T + len(ROUTES) * T + p * NK * T + k * T + t
```

</details>

---

## Part 2: S3 Capacity Constraints (25 min)

### Context

Spot supplier S3 has limited capacity: **150,000 L per month** total across all plants.

In our model, S3 is supplier index `s=2`. The constraint for month `t` is:

```
x[0, 2, t] + x[1, 2, t] + x[2, 2, t] ≤ 150,000
```

We need 12 such constraints (one per month).

### Your task

Build two arrays:
- **A_ub**: shape (12, 252) — coefficient matrix
- **b_ub**: shape (12,) — right-hand side (capacity limit)

The LP solver will enforce `A_ub @ x ≤ b_ub`.

**Hint:** 
- Row `t` of `A_ub` should have 1.0 at positions `idx_x(p, 2, t)` for p=0,1,2
- All other entries are 0
- Each entry of `b_ub` is the capacity limit

In [ ]:
# YOUR CODE HERE


def build_inequality_constraints(s3_cap: float):
    """Build S3 monthly capacity cap: Σ_p x[p, 2, t] ≤ s3_cap."""
    raise NotImplementedError("Implement build_inequality_constraints() function.")

### Test your implementation

In [ ]:
# Build constraints
A_ub, b_ub = build_inequality_constraints(150_000)

# Shape checks
assert A_ub.shape == (12, 252), f"A_ub shape should be (12, 252), got {A_ub.shape}"
assert b_ub.shape == (12,), f"b_ub shape should be (12,), got {b_ub.shape}"
print("✅ Matrix dimensions correct")

# Right-hand side check
assert np.all(b_ub == 150_000), "All b_ub entries should equal s3_cap"
print("✅ Right-hand side correct")

# Coefficient check - each row should have exactly 3 ones (one per plant)
for t in range(T):
    row = A_ub[t, :]
    assert np.sum(row) == 3.0, f"Row {t} should sum to 3.0 (one per plant)"

    # Check that the ones are at the correct positions
    for p in range(NP):
        expected_idx = idx_x(p, 2, t)  # s=2 is S3
        assert row[expected_idx] == 1.0, f"Row {t} should have 1.0 at index {expected_idx}"

    # Check that all other entries are 0
    nonzero_indices = np.where(row != 0)[0]
    assert len(nonzero_indices) == 3, f"Row {t} should have exactly 3 non-zero entries"

print("✅ All constraints correctly encode S3 capacity limits")
print(f"\nSparsity: {np.sum(A_ub != 0)} non-zero entries out of {A_ub.size} total")
print(f"           ({100 * np.sum(A_ub != 0) / A_ub.size:.1f}% non-zero)")

### Visualize the constraint matrix

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a more informative visualization
fig, ax = plt.subplots(figsize=(14, 5))

# Use heatmap for better visibility
sns.heatmap(
    A_ub, cmap="YlOrRd", cbar_kws={"label": "Coefficient"}, linewidths=0, ax=ax, vmin=0, vmax=1
)

# Add vertical lines to separate variable blocks
ax.axvline(x=108, color="blue", linewidth=2, linestyle="--", alpha=0.7, label="x | u boundary")
ax.axvline(x=180, color="green", linewidth=2, linestyle="--", alpha=0.7, label="u | e boundary")

# Annotate variable blocks at the top
ax.text(
    54,
    -1.5,
    "x[p,s,t]\n(spot purchases)",
    ha="center",
    va="bottom",
    fontsize=10,
    fontweight="bold",
    color="darkred",
)
ax.text(
    144,
    -1.5,
    "u[r,t]\n(transfers)",
    ha="center",
    va="bottom",
    fontsize=10,
    fontweight="bold",
    color="darkblue",
)
ax.text(
    216,
    -1.5,
    "e[p,k,t]\n(excess sales)",
    ha="center",
    va="bottom",
    fontsize=10,
    fontweight="bold",
    color="darkgreen",
)

# Labels and formatting
ax.set_xlabel("Variable Index", fontsize=11, fontweight="bold")
ax.set_ylabel("Month", fontsize=11, fontweight="bold")
ax.set_title(
    "S3 Capacity Constraint Matrix (12 constraints × 252 variables)",
    fontsize=12,
    fontweight="bold",
    pad=20,
)
ax.set_yticks(np.arange(12) + 0.5)
ax.set_yticklabels(
    ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"], rotation=0
)

# Add x-axis ticks for key positions
ax.set_xticks([0, 107, 108, 179, 180, 251])
ax.set_xticklabels(["0", "107", "108", "179", "180", "251"])

ax.legend(loc="upper right", framealpha=0.9)
plt.tight_layout()
plt.show()

# Summary stats
nonzero = np.sum(A_ub != 0)
print("📊 Matrix structure:")
print(f"   • Only {nonzero}/3,024 entries are non-zero ({100 * nonzero / A_ub.size:.1f}%)")
print("   • Each month constrains exactly 3 variables (one per plant)")
print("   • S3 purchases are in columns 2, 14, 26, ... (every 12th from col 2)")

<details>
<summary><b>💡 SOLUTION</b> (click to expand)</summary>

```python
def build_inequality_constraints(s3_cap: float):
    """Build S3 monthly capacity cap: Σ_p x[p, 2, t] ≤ s3_cap."""
    N_VARS = 252
    A_ub = np.zeros((T, N_VARS))
    b_ub = np.full(T, s3_cap)
    
    for t in range(T):
        for p in range(NP):
            A_ub[t, idx_x(p, 2, t)] = 1.0  # s=2 is S3
    
    return A_ub, b_ub
```

</details>

---

## Part 3: Solve and Interpret (20 min)

### Now let's solve the full LP!

The other constraint matrix (`build_equality_constraints` for mass balance) is pre-implemented. We'll use your indexing functions to solve the complete problem.

In [ ]:
# Monkey-patch our implementations into the module
import milk_balancing.lp_model as lp_model

lp_model.idx_x = idx_x
lp_model.idx_u = idx_u
lp_model.idx_e = idx_e
lp_model.build_inequality_constraints = build_inequality_constraints

from milk_balancing import load_data, solve_scenario

# Solve baseline scenario
data = load_data()
result = solve_scenario(data)

if "error" in result:
    print("❌ Solver failed:")
    for err in result["error"]:
        print(f"  - {err}")
else:
    print("✅ LP solved successfully!\n")
    print(f"Total cost: €{result['total_cost']:,.0f}")
    print(f"  Farm:      €{result['farm_cost']:,.0f}")
    print(f"  Spot:      €{result['spot_cost']:,.0f}")
    print(f"  Transport: €{result['trans_cost']:,.0f}")
    print(f"  Sales rev: €{result['sales_rev']:,.0f}")

### Interpreting Shadow Prices

The **shadow price** of a constraint tells you how much the optimal cost would change if you relaxed that constraint by one unit.

For S3 capacity:
- Positive shadow price → S3 cap is binding (we're hitting the limit)
- Zero shadow price → S3 cap is slack (we're not using full capacity)

Let's look at which months hit the S3 limit:

In [ ]:
from milk_balancing.constants import MONTHS

duals_s3 = result["duals_s3"]

print("S3 Shadow Prices (€/L):")
print("Month     Shadow Price    Interpretation")
print("-" * 55)
for t, month in enumerate(MONTHS):
    sp = duals_s3[t]
    status = "🔴 BINDING" if sp > 0.001 else "  slack"
    print(f"{month:8s}  {sp:12.4f}    {status}")

# Show actual S3 usage
x_sol = result["x_sol"]  # Shape: (NP, NS, T)
s3_usage = x_sol[:, 2, :].sum(axis=0)  # Total S3 purchases per month (s=2)

print("\nS3 Usage vs Capacity:")
for t, month in enumerate(MONTHS):
    pct = 100 * s3_usage[t] / 150_000
    bar = "█" * int(pct / 5)
    print(f"{month:8s}  {s3_usage[t] / 1000:6.1f} kL / 150 kL  ({pct:5.1f}%)  {bar}")

### Experiment: Relax the S3 cap

Shadow price theory says: if S3 constraint has shadow price λ, then relaxing the cap by ΔC should reduce cost by approximately λ × ΔC.

Let's test this:

In [ ]:
# Pick a month where S3 is binding
binding_months = [(t, MONTHS[t], duals_s3[t]) for t in range(T) if duals_s3[t] > 0.001]

if binding_months:
    t_test, month_name, shadow_price = binding_months[0]
    print(f"Testing shadow price for {month_name}: {shadow_price:.4f} €/L\n")

    # Relax S3 cap by 10,000 L
    delta_cap = 10_000
    result_relaxed = solve_scenario(data, s3_cap=150_000 + delta_cap)

    actual_savings = result["total_cost"] - result_relaxed["total_cost"]
    predicted_savings = shadow_price * delta_cap

    print(f"Increased S3 cap by {delta_cap:,} L\n")
    print(f"Predicted cost reduction: €{predicted_savings:,.2f}")
    print(f"Actual cost reduction:    €{actual_savings:,.2f}")
    print(f"Error: {abs(actual_savings - predicted_savings) / predicted_savings * 100:.1f}%")

    if abs(actual_savings - predicted_savings) < 10:
        print("\n✅ Shadow price prediction is accurate!")
else:
    print("S3 constraint is not binding in any month — plenty of slack capacity")

### Experiment: What if S3 becomes cheaper?

Try reducing S3 spot price by 20% — does S3 usage increase?

In [ ]:
result_cheap_s3 = solve_scenario(data, spot_mult=[1.0, 1.0, 0.8])  # 20% discount on S3

s3_usage_cheap = result_cheap_s3["x_sol"][:, 2, :].sum(axis=0)

print("S3 Usage Comparison (baseline vs 20% price cut):\n")
print("Month     Baseline    Cheap S3    Change")
print("-" * 50)
for t, month in enumerate(MONTHS):
    change = s3_usage_cheap[t] - s3_usage[t]
    arrow = "↑" if change > 100 else " "
    print(
        f"{month:8s}  {s3_usage[t] / 1000:7.1f} kL  {s3_usage_cheap[t] / 1000:7.1f} kL  {change / 1000:+7.1f} kL {arrow}"
    )

print(f"\nTotal cost reduction: €{result['total_cost'] - result_cheap_s3['total_cost']:,.0f}")

---

## 🎉 Congratulations!

You've implemented the core of an LP model and solved a real supply chain optimization problem.

### Key takeaways

1. **Variable indexing** is the foundation — get this right and the rest follows
2. **Sparse matrices** are beautiful — 99% of A_ub is zeros
3. **Shadow prices** connect math to economics — they're the marginal value of resources

### Optional: Run the full test suite

To verify your implementation against production tests:

```bash
pytest tests/test_lp_model.py -v
```

### Next steps

- Try implementing `build_equality_constraints()` (the 36 balance constraints)
- Explore the Streamlit dashboard: `streamlit run app.py`
- Check out the complete solution on the `main` branch